In [2]:
import datetime
import functools
import logging
import operator
import orjson
import io
import json

import pyarrow as pa
import pyarrow.compute as pc
from pyarrow import csv as pacsv
import pyarrow.dataset as ds
import time
from deltalake import DeltaTable
from deltalake.exceptions import TableNotFoundError

from azure.core.exceptions import ClientAuthenticationError
from azure.functions import Blueprint, AuthLevel, HttpRequest, HttpResponse, FunctionApp
from azure.identity import DefaultAzureCredential

_STORAGE_SCOPE = "https://storage.azure.com/.default"
_CREDENTIAL = DefaultAzureCredential()

def load_study(study) -> DeltaTable | HttpResponse:
    table_uri = f"abfs://{study}/datums"
    account_name = "trailsoutputs"

    try:
        storage_options = {"ACCOUNT_NAME": account_name, "BEARER_TOKEN": _CREDENTIAL.get_token(_STORAGE_SCOPE).token}
        dt = DeltaTable(table_uri, storage_options=storage_options)

    except ClientAuthenticationError:
        logging.exception(
            "Could not acquire a storage token for account '%s'. Check that the function app has a "
            "managed identity enabled and that AZURE_CLIENT_ID is set if it is user-assigned.",
            account_name
        )
        return HttpResponse(status_code=500)
    except TableNotFoundError:
        logging.warning("No delta table at %s (account '%s')", table_uri, account_name)
        return HttpResponse(f"No data for study '{study}'", status_code=404)
    except Exception:
        # Auth/permission failures from the storage account surface here as OSError,
        # so log the detail; the identity most likely lacks Storage Blob Data Reader.
        logging.exception("Failed to open %s on account '%s'", table_uri, account_name)
        return HttpResponse(status_code=500)

    return dt

In [55]:
study_response = load_study("mtm-t2")
if isinstance(study_response, HttpResponse): raise Exception("oops")

dataset = study_response.to_pyarrow_dataset()

days=7

flows_per_day = {}

today = datetime.datetime.now().astimezone()
offset = today.utcoffset().total_seconds() if today.utcoffset() else 0  # type: ignore 
today = datetime.datetime(year=today.year, month=today.month, day=today.day) + datetime.timedelta(days=1)
for i in range(1, days+1):
    date = datetime.datetime.strftime(today - datetime.timedelta(days=i), "%Y-%m-%d")
    flows_per_day[date] = 0
start = today - datetime.timedelta(days=int(days)+1) # type: ignore
start = datetime.datetime(year=start.year, month=start.month, day=start.day)
ts_start = start.timestamp()

tbl = dataset.to_table(
    columns={
        "day": (pc.field("ts") - offset).cast(pa.int64(), safe=False).cast(pa.timestamp("s")).cast(pa.date32()),
        "data": pc.field("data")
    },
    filter= (pc.field("ts") >= ts_start) & (pc.field("type")=="Flow")
)
tbl = tbl.filter(pc.match_substring(tbl["data"], '\"value\":\"Completion\"'))
# print(tbl)
counts = tbl.group_by("day").aggregate([("day", "count")])


days_col, counts_col = counts.column("day"), counts.column("day_count")

for i in range(len(days_col)):
    date = str(days_col[i])
    if date in flows_per_day:
        flows_per_day[date] = int(counts_col[i])

flows_per_day

{'2026-08-12': 0,
 '2026-08-11': 0,
 '2026-08-10': 0,
 '2026-08-09': 0,
 '2026-08-08': 1,
 '2026-08-07': 0,
 '2026-08-06': 0}

In [148]:
# response = load_study("mtm-t2")
# if isinstance(response, HttpResponse): raise Exception("Something went wrong")
# days = 7

# now = datetime.datetime.now(datetime.timezone.utc).astimezone()
# start = now - datetime.timedelta(days=days) 
# start = datetime.datetime(year=start.year, month=start.month, day=start.day)
# ts_end = now.timestamp()
# ts_start = start.timestamp()
# print(ts_start)


# # count 'data' of type 'Flow' that contains the word 'completion' do this filter first
# # - then transform ts into days
# # aggregate by day - do this last
# dataset = response.to_pyarrow_dataset()

# subtable = dataset.to_table(
#     columns=["ts","data"],
#     filter=(pc.field("ts") >= ts_start) & (pc.field("type") == "Flow")
# )

# condition = (pc.match_substring(pc.field("data"), pattern='\"value\":\"Completion\"'))

# subtable.filter(condition).num_rows


In [ ]:
study = 'mtm-t2'
tipe = "CompletedFlow"
ts_start = 0


study_response = load_study(study)
if isinstance(study_response, HttpResponse):
    raise Exception("oops")

now = datetime.datetime.now(datetime.timezone.utc).astimezone()
start = now - datetime.timedelta(days=days) # type: ignore
start = datetime.datetime(year=start.year, month=start.month, day=start.day)
ts_start = start.timestamp()

dataset = study_response.to_pyarrow_dataset()

# adding a special type here to avoid writing a whole new function for completed flows
if tipe == "CompletedFlow":
    subtable = dataset.to_table(
        columns=["ts","data"],
        filter=(pc.field("ts") >= ts_start) & (pc.field("type") == "Flow")
    )

    condition = (pc.match_substring(pc.field("data"), pattern='\"value\":\"Completion\"'))
    return subtable.filter(condition).num_rows


else:
    subtable = dataset.to_table(
        columns=["ts","data"],
        filter=(pc.field("ts") >= ts_start) & (pc.field("type") == tipe)
    )
    return subtable.num_rows


1